# Week 2 Day 4 — AI Dashboard Components

Create modular building blocks for news sentiment and a simple assistant that can answer stock- and portfolio-related questions.

## Objective

Show how the dashboard could retrieve recent headlines, summarize them, estimate sentiment, and answer simple questions using a rule-based assistant.

In [ ]:
import re
from typing import List, Dict

import pandas as pd

news_corpus = {
    'ABUK': [
        'ABUK reports strong quarterly sales growth',
        'Analysts remain bullish after management guidance',
        'The stock sees renewed interest after positive earnings',
    ],
    'HRHO': [
        'HRHO faces margin pressure despite stable revenue',
        'Analysts note cautious sentiment around near-term demand',
        'The company is under watch after weaker guidance',
    ],
}

def retrieve_news(symbol: str, limit: int = 3) -> List[str]:
    return news_corpus.get(symbol, ['No recent news available'])[:limit]

def summarize_news(headlines: List[str]) -> str:
    return ' | '.join(headlines)

def estimate_sentiment(headlines: List[str]) -> Dict[str, float]:
    text = ' '.join(headlines).lower()
    positive_words = ['strong', 'bullish', 'positive', 'growth', 'renewed']
    negative_words = ['pressure', 'cautious', 'weaker', 'negative']
    pos_score = sum(text.count(word) for word in positive_words)
    neg_score = sum(text.count(word) for word in negative_words)
    score = (pos_score - neg_score) / max(1, pos_score + neg_score)
    if score > 0.1:
        label = 'Positive'
    elif score < -0.1:
        label = 'Negative'
    else:
        label = 'Neutral'
    return {'score': score, 'label': label}

def build_sentiment_table(symbol: str) -> pd.DataFrame:
    headlines = retrieve_news(symbol)
    sentiment = estimate_sentiment(headlines)
    rows = []
    for headline in headlines:
        rows.append({'headline': headline, 'summary': headline, 'sentiment': sentiment['label'], 'confidence': round(abs(sentiment['score']), 2)})
    return pd.DataFrame(rows)

def chat_response(question: str, symbol: str = 'ABUK', metrics: Dict[str, float] | None = None) -> str:
    q = question.lower()
    if 'sentiment' in q:
        return f"{symbol} sentiment is {estimate_sentiment(retrieve_news(symbol))['label']} based on the latest headlines."
    if 'portfolio' in q or 'performance' in q:
        return f"The portfolio currently shows a return of {metrics.get('portfolio_return', 0.0):.2%} and drawdown {metrics.get('drawdown', 0.0):.2%}." if metrics else 'Portfolio metrics were not provided.'
    if 'strategy' in q:
        return 'The strategy currently uses predicted return signals converted into long-only portfolio weights.'
    return 'I can help with stock sentiment, portfolio performance, strategy details, drawdown, returns, and benchmark comparisons.'

symbol = 'ABUK'
print(build_sentiment_table(symbol))
print(chat_response('How is the portfolio performing?', symbol, {'portfolio_return': 0.12, 'drawdown': 0.04}))

## Conclusion

This notebook exposes modular functions for news retrieval, summarization, sentiment scoring, and a simple question-answering layer that can be wired into a dashboard later.